In [72]:
#!pip install -U openai tqdm
!pip install azure-identity

In [73]:
import os
from google.colab import userdata
from openai import AzureOpenAI
from azure.identity import DefaultAzureCredential

RESOURCE_GROUP = userdata.get('RESOURCE_GROUP').strip()
OPENAI_API_KEY = userdata.get('OPENAI_API_KEY').strip()

OPENAI_ENDPOINT = f"https://{RESOURCE_GROUP}.openai.azure.com"
SUBSCRIPTION_ID = userdata.get('SUBSCRIPTION_ID')

os.environ["AZURE_SUBSCRIPTION_ID"] = SUBSCRIPTION_ID
os.environ["AZURE_RESOURCE_GROUP"]  = "CIS-5270"
os.environ["AZURE_AOAI_ACCOUNT"]    = RESOURCE_GROUP
os.environ["AZURE_OPENAI_API_KEY"]  = OPENAI_API_KEY
os.environ["AZURE_OPENAI_ENDPOINT"] = OPENAI_ENDPOINT

CREDENTIAL = DefaultAzureCredential()

openai_client = AzureOpenAI(
    api_key=OPENAI_API_KEY,
    azure_endpoint=OPENAI_ENDPOINT,
    api_version="2025-04-01-preview",
)

BASE_MODEL = "gpt-4.1-nano-2025-04-14"

BASE_DEPLOYMENT = "gpt-4.1-nano"

print("Connected to Azure OpenAI")

Connected to Azure OpenAI


In [74]:
#Job Checking: Are the finetuning jobs finished?
# Job ID
#c4: ftjob-e163528e6cf34e8bb6abe72dc7b1de08
#c3: ftjob-fd0b7badeffd45c4acdd5641a4c659ed
#c2: ftjob-ac11598ad2c14758b395bc5860c18043
#c1: ftjob-58abe7a24f934eb68e5040dbb8010565
job_id = "ftjob-58abe7a24f934eb68e5040dbb8010565"

final_job = openai_client.fine_tuning.jobs.retrieve(job_id)

print("Status:", final_job.status)
print("Fine-tuned model:", final_job.fine_tuned_model)
print("Error:", final_job.error)

Status: succeeded
Fine-tuned model: gpt-4.1-nano-2025-04-14.ft-58abe7a24f934eb68e5040dbb8010565-dpo_main
Error: None


In [75]:
# Cancelling unused jobs
'''
cancelled = openai_client.fine_tuning.jobs.cancel("ftjob-fd0b7badeffd45c4acdd5641a4c659ed")

print("Status:", cancelled.status)
'''


'\ncancelled = openai_client.fine_tuning.jobs.cancel("ftjob-fd0b7badeffd45c4acdd5641a4c659ed")\n\nprint("Status:", cancelled.status)\n'

# Direct Preference Optimization

Fine-tune GPT-4.1-nano on preference pairs of the form:

- prompt = FEVER passage + claim
- chosen = better answer (correct label + grounded justification)
- rejected = worse answer (wrong label or flawed justification)

1. converting the DPO data into Azure format
2. uploading the training file
3. submitting the DPO fine-tuning job
4. monitoring job status
5. preparing comparable dev-set evaluation
6. stopping before deployment

## Step 1: Load DPO Training Data

We start from `dpo_data_3k.jsonl`, which already contains the final preference pairs:

- `prompt`
- `chosen`
- `rejected`

Unlike SFT, DPO does not use standard `{"messages": [...]}` training rows.
Instead, Azure expects each row to contain:

- `input`
- `preferred_output`
- `non_preferred_output`

Here, we convert the dataset into that format below.

In [76]:
#imports, seed, helpers
import json
import os
import time
import random
import re
from collections import Counter, defaultdict
from tqdm import tqdm

random.seed(42)
def load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

def save_jsonl(path, rows):
    with open(path, "w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

In [77]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [78]:
data_dir = "/content/drive/MyDrive"


dpo_path = f"{data_dir}/dpo_data_3k.jsonl"
dev_path = f"{data_dir}/fever_dev_joined.jsonl"

print("DPO path exists:", os.path.exists(dpo_path))
print("DEV path exists:", os.path.exists(dev_path))

dpo_data = load_jsonl(dpo_path)
dev_data = load_jsonl(dev_path)

print("DPO rows:", len(dpo_data))
print("Dev rows:", len(dev_data))
print("DPO example keys:", dpo_data[0].keys())
print("Dev example keys:", dev_data[0].keys())

DPO path exists: True
DEV path exists: True
DPO rows: 3000
Dev rows: 19891
DPO example keys: dict_keys(['id', 'prompt', 'chosen', 'rejected'])
Dev example keys: dict_keys(['claim', 'passage', 'label', 'id'])


## Step 2: Define the Matched Prompt Format

To keep DPO evaluation comparable to the baseline and SFT notebooks,
we use the same fact-checking format:

- output one label
- then one sentence justification
- labels must be exactly:
  - SUPPORTED
  - CONTRADICTED
  - NOT MENTIONED

We keep this instruction in the DPO training examples so the learned behavior matches our evaluation setup.

In [79]:
PROMPT_MATCHED_SYSTEM = """You are a fact-checking assistant. Given a passage and a claim, respond with the verdict followed by a one-sentence justification quoting or closely paraphrasing the passage.
Format: LABEL: justification sentence
Label must be one of: SUPPORTED, CONTRADICTED, NOT MENTIONED"""

## Step 3: Convert DPO Data into Azure DPO Format

Each row is converted into:

- `input.messages`
- `preferred_output`
- `non_preferred_output`

We also remove the original `id` field and reshapes the example into the schema Azure expects for DPO.

In [80]:
def convert_row_to_dpo_format(row):
    return {
        "input": {
            "messages": [
                {
                    "role": "system",
                    "content": PROMPT_MATCHED_SYSTEM
                },
                {
                    "role": "user",
                    "content": row["prompt"]
                }
            ]
        },
        "preferred_output": [
            {
                "role": "assistant",
                "content": row["chosen"]
            }
        ],
        "non_preferred_output": [
            {
                "role": "assistant",
                "content": row["rejected"]
            }
        ]
    }

converted_rows = [convert_row_to_dpo_format(row) for row in dpo_data]

print("Converted rows:", len(converted_rows))
print(json.dumps(converted_rows[0], indent=2)[:1500])

Converted rows: 3000
{
  "input": {
    "messages": [
      {
        "role": "system",
        "content": "You are a fact-checking assistant. Given a passage and a claim, respond with the verdict followed by a one-sentence justification quoting or closely paraphrasing the passage.\nFormat: LABEL: justification sentence\nLabel must be one of: SUPPORTED, CONTRADICTED, NOT MENTIONED"
      },
      {
        "role": "user",
        "content": "Passage: The law of value -LRB- German : Wertgesetz -RRB- is a central concept in Karl Marx 's critique of political economy , first expounded in his polemic The Poverty of Philosophy -LRB- 1847 -RRB- against Pierre-Joseph Proudhon , with reference to David Ricardo 's economics.See Marx , The Poverty of Philosophy , chapter 1 part 2 where Marx refers to Proudhon 's own `` law of value '' and chapter 3 , titled `` Application of the Law of the Proportionality of Value '' .\n\nClaim: Law of value is a peripheral concept in Karl Marx's critique of pol

## Step 4: Basic Local Cleaning

Remove rows that are obviously unsafe for training:

- missing messages
- empty strings
- chosen = rejected

Sanity check before Azure preprocessing.

In [81]:
cleaned_rows = []

for row in converted_rows:
    try:
        system_msg = row["input"]["messages"][0]["content"].strip()
        user_msg = row["input"]["messages"][1]["content"].strip()
        chosen = row["preferred_output"][0]["content"].strip()
        rejected = row["non_preferred_output"][0]["content"].strip()
    except Exception:
        continue

    if not system_msg or not user_msg or not chosen or not rejected:
        continue
    if chosen == rejected:
        continue

    cleaned_rows.append(row)

print("Usable DPO rows:", len(cleaned_rows))
print("Dropped rows:", len(converted_rows) - len(cleaned_rows))

Usable DPO rows: 3000
Dropped rows: 0


## Step 5: Save Azure Upload File

Unlike SFT, we are not using a validation split or hyperparameter sweep here.
We train on the full cleaned DPO dataset and evaluate later on a separate held-out FEVER dev set.

In [82]:
save_jsonl("dpo_train_azure.jsonl", cleaned_rows)
print("Saved dpo_train_azure.jsonl with", len(cleaned_rows), "rows")

Saved dpo_train_azure.jsonl with 3000 rows


## Step 6: Upload the Training File

Azure fine-tuning requires the training file to be uploaded first.
When uploaded, we have Azure assign a file ID that is used when creating the DPO job.

In [83]:
print("Uploading DPO training file...")
with open("dpo_train_azure.jsonl", "rb") as f:
    train_file = openai_client.files.create(file=f, purpose="fine-tune")

train_file_id = train_file.id
print("Training file ID:", train_file_id)

Uploading DPO training file...
Training file ID: file-f748449fe30f46d0965316777426f71d


## Step 7: Submit the DPO Fine-Tuning Job

We use one DPO configuration:

- lr multiplier = 1.0
- epochs = 1
- batch size = 1

This mirrors the “simple baseline config” style used in the SFT notebook.

In [84]:
# RUNNING DPO JOB
'''
print(f"Creating DPO fine-tuning job for {BASE_MODEL}")

job = openai_client.fine_tuning.jobs.create(
    training_file=train_file_id,
    model=BASE_MODEL,
    method={
        "type": "dpo",
        "dpo": {
            "hyperparameters": {
                "n_epochs": 1,
                "batch_size": 1,
                "learning_rate_multiplier": 1.0
            }
        }
    },
    extra_body={"trainingType": "GlobalStandard"},
    suffix="dpo_main"
)

print("Job ID:", job.id)
print("Initial status:", job.status)
'''

'\nprint(f"Creating DPO fine-tuning job for {BASE_MODEL}")\n\njob = openai_client.fine_tuning.jobs.create(\n    training_file=train_file_id,\n    model=BASE_MODEL,\n    method={\n        "type": "dpo",\n        "dpo": {\n            "hyperparameters": {\n                "n_epochs": 1,\n                "batch_size": 1,\n                "learning_rate_multiplier": 1.0\n            }\n        }\n    },\n    extra_body={"trainingType": "GlobalStandard"},\n    suffix="dpo_main"\n)\n\nprint("Job ID:", job.id)\nprint("Initial status:", job.status)\n'

## Step 8: Monitor the DPO Job

In [85]:
'''
def wait_for_job(job_id, poll_seconds=30):
    while True:
        current = openai_client.fine_tuning.jobs.retrieve(job_id)
        print(f"{job_id}: {current.status}")

        if current.status in {"succeeded", "failed", "cancelled"}:
            return current

        time.sleep(poll_seconds)

final_job = wait_for_job(job.id, poll_seconds=30)
'''

'\ndef wait_for_job(job_id, poll_seconds=30):\n    while True:\n        current = openai_client.fine_tuning.jobs.retrieve(job_id)\n        print(f"{job_id}: {current.status}")\n\n        if current.status in {"succeeded", "failed", "cancelled"}:\n            return current\n\n        time.sleep(poll_seconds)\n\nfinal_job = wait_for_job(job.id, poll_seconds=30)\n'

In [86]:
print("Final job status:", final_job.status)
print("Model:", getattr(final_job, "model", None))
print("Fine-tuned model:", getattr(final_job, "fine_tuned_model", None))
print("Error:", getattr(final_job, "error", None))
print("Training file:", getattr(final_job, "training_file", None))

Final job status: succeeded
Model: gpt-4.1-nano-2025-04-14
Fine-tuned model: gpt-4.1-nano-2025-04-14.ft-58abe7a24f934eb68e5040dbb8010565-dpo_main
Error: None
Training file: file-1e708d53efea4367a392f64f99cbadeb


## Step 9: Load the FEVER Dev Set

To keep evaluation comparable to the baseline and SFT notebooks,
we use the same dev evaluation recipe:

- load FEVER dev
- stratify by label
- sample 667 examples per class
- total evaluation set = 2001 examples

In [87]:
# Zheng code
random.seed(42)
buckets = defaultdict(list)

for ex in dev_data:

    buckets[ex["label"]].append(ex)

eval_sample = []

for lbl, items in buckets.items():

    random.shuffle(items)

    eval_sample.extend(items[:667])

random.shuffle(eval_sample)

print(f"dev sample: {len(eval_sample)} examples")

print(Counter(ex["label"] for ex in eval_sample))

dev sample: 2001 examples
Counter({'SUPPORTED': 667, 'CONTRADICTED': 667, 'NOT MENTIONED': 667})


## Step 10: Define Label Extraction and Evaluation Helpers

To compare baseline vs DPO fairly, we parse model outputs in the same way:

- expected format: `LABEL: justification`
- fallback: exact label only
- fallback: whole-word label scan

In [88]:
VALID_LABELS = {"SUPPORTED", "CONTRADICTED", "NOT MENTIONED"}

def extract_label(text):
    if not isinstance(text, str):
        return None

    for lbl in VALID_LABELS:
        if text.upper().startswith(lbl + ":"):
            return lbl

    if text.strip() in VALID_LABELS:
        return text.strip()

    for lbl in sorted(VALID_LABELS, key=len, reverse=True):
        if re.search(rf"\b{re.escape(lbl)}\b", text.upper()):
            return lbl

    return None

In [89]:
def predict(passage, claim, deployment_name, system_prompt):
    user_msg = f"Passage: {passage}\n\nClaim: {claim}"

    try:
        response = openai_client.chat.completions.create(
            model=deployment_name,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_msg},
            ],
            temperature=0.0,
            max_tokens=100,
        )
        raw = response.choices[0].message.content.strip()
        pred_label = extract_label(raw)
        return pred_label, raw
    except Exception as e:
        return None, f"ERROR: {e}"

In [90]:
def run_eval(deployment_name, examples, out_path):
    done_ids = set()
    results = []

    if os.path.exists(out_path):
        with open(out_path, "r", encoding="utf-8") as f:
            for line in f:
                r = json.loads(line)
                done_ids.add(r["id"])
                results.append(r)
        print(f"resuming, {len(done_ids)} already done")

    skipped = 0
    with open(out_path, "a", encoding="utf-8") as out_f:
        for i, ex in enumerate(examples):
            if ex["id"] in done_ids:
                continue

            pred_label, raw = predict(
                ex["passage"],
                ex["claim"],
                deployment_name,
                PROMPT_MATCHED_SYSTEM
            )

            if pred_label is None:
                skipped += 1

            record = {
                "id": ex["id"],
                "label": ex["label"],
                "pred_label": pred_label,
                "raw": raw,
            }

            out_f.write(json.dumps(record, ensure_ascii=False) + "\n")
            out_f.flush()
            results.append(record)

            if (i + 1) % 200 == 0:
                print(f"[{i+1}/{len(examples)}] skipped={skipped}")

    print(f"done. skipped={skipped}")
    return results

In [91]:
test_ex = eval_sample[0]

pred_label, raw = predict(
    test_ex["passage"],
    test_ex["claim"],
    BASE_DEPLOYMENT,
    PROMPT_MATCHED_SYSTEM
)

print("pred_label:", pred_label)
print("raw:", raw)

pred_label: None
raw: ERROR: Error code: 404 - {'error': {'message': 'Could not find an existing deployment to match the model in the request. Please verify the model matches an existing deployment in the account.', 'type': 'invalid_request_error', 'param': None, 'code': None}}


In [92]:
print("BASE_DEPLOYMENT:", BASE_DEPLOYMENT)

BASE_DEPLOYMENT: gpt-4.1-nano


## Step 11: Baseline Evaluation on Dev

Before comparing the fine-tuned DPO model, we score the untuned base deployment on the same sampled dev set.
This gives us the “before fine-tuning” number for comparison.

In [93]:
'''
baseline_results_path = "dpo_baseline_dev.jsonl"

baseline_results = run_eval(
    deployment_name=BASE_DEPLOYMENT,
    examples=eval_sample,
    out_path=baseline_results_path,
)
'''

'\nbaseline_results_path = "dpo_baseline_dev.jsonl"\n\nbaseline_results = run_eval(\n    deployment_name=BASE_DEPLOYMENT,\n    examples=eval_sample,\n    out_path=baseline_results_path,\n)\n'

## Step 12: Metric Reporting

We report:

- overall accuracy
- macro accuracy
- per-class accuracy
- confusion matrix

These are the same style of metrics used in the SFT notebook.

In [ ]:
def compute_metrics(results):
    labels = ["SUPPORTED", "CONTRADICTED", "NOT MENTIONED"]
    per_class = {lbl: {"correct": 0, "total": 0} for lbl in labels}
    confusion = {true: {pred: 0 for pred in labels} for true in labels}
    total_correct = 0

    for r in results:
        gt, pred = r["label"], r["pred_label"]
        per_class[gt]["total"] += 1
        if pred in labels:
            confusion[gt][pred] += 1
        if pred == gt:
            per_class[gt]["correct"] += 1
            total_correct += 1

    macro_acc = sum(
        per_class[lbl]["correct"] / per_class[lbl]["total"]
        for lbl in labels if per_class[lbl]["total"] > 0
    ) / len(labels)

    overall_acc = total_correct / len(results)

    print(f"  overall accuracy: {overall_acc:.3f}  ({total_correct}/{len(results)})")
    print(f"  macro accuracy:   {macro_acc:.3f}")
    print()
    print("  Per-class accuracy:")
    for lbl in labels:
        c = per_class[lbl]
        acc = c["correct"] / c["total"] if c["total"] else 0
        print(f"    {lbl:20s}: {acc:.3f} ({c['correct']}/{c['total']})")
    print()
    print("  Confusion matrix (rows=true, cols=pred):")
    col_w = 14
    header = " " * 22 + "".join(f"{lbl[:col_w]:>{col_w}}" for lbl in labels)
    print(header)
    for true_lbl in labels:
        row = f"  {true_lbl:20s}" + "".join(
            f"{confusion[true_lbl][pred_lbl]:>{col_w}}" for pred_lbl in labels
        )
        print(row)

    return {
        "overall_accuracy": overall_acc,
        "macro_accuracy": macro_acc,
    }

In [94]:
print("Baseline Results")
baseline_metrics = compute_metrics(baseline_results)

Baseline Results


NameError: name 'baseline_results' is not defined

## Step 13: Stop Before Deployment

The DPO training job may have produced a fine-tuned model ID.

The code for deployment and post-deployment evaluation is included below,
but should not be run yet.

In [ ]:
if final_job.status == "succeeded":
    WINNING_FINE_TUNED_MODEL = final_job.fine_tuned_model
    print("Fine-tuned model ready:", WINNING_FINE_TUNED_MODEL)
else:
    WINNING_FINE_TUNED_MODEL = None
    print("No deployable fine-tuned model because training did not succeed.")

print("\nSTOP: request course staff permission before deployment.")

Fine-tuned model ready: gpt-4.1-nano-2025-04-14.ft-58abe7a24f934eb68e5040dbb8010565-dpo_main

STOP: request course staff permission before deployment.


# After permission is granted

## Step 14: Deploy the Fine-Tuned DPO Model

only to be run after seewon approval

In [ ]:
DPO_DEPLOYMENT = "1-nano-2025-04-14-dpo_main"
dpo_results = run_eval(

    deployment_name=DPO_DEPLOYMENT,

    examples=eval_sample,

    out_path="dpo_finetuned_dev.jsonl",

)

[200/2001] skipped=21
[400/2001] skipped=40
[600/2001] skipped=61
[800/2001] skipped=79
[1000/2001] skipped=96
[1200/2001] skipped=120


KeyboardInterrupt: 

In [ ]:
# Load whatever has been saved so far
DPO_RESULTS_PATH = "dpo_finetuned_dev.jsonl"
dpo_partial_results = load_jsonl(DPO_RESULTS_PATH)

print("Saved DPO results:", len(dpo_partial_results))
print("Skipped:", sum(r["pred_label"] is None for r in dpo_partial_results))

# Use first 1000 completed rows
EVAL_N = 1000
dpo_eval_subset = dpo_partial_results[:EVAL_N]
baseline_eval_subset = baseline_results[:EVAL_N]

print("Using examples:", len(dpo_eval_subset))

Saved DPO results: 1201
Skipped: 120
Using examples: 1000


## Step 15: Evaluate the Fine-Tuned DPO Model on Dev

We now evaluate the DPO model on the exact same stratified dev sample used for the baseline.
This keeps the comparison fair.

In [ ]:
print("DPO Fine-Tuned Results, first 1000")
dpo_metrics_1000 = compute_metrics(dpo_eval_subset)

DPO Fine-Tuned Results, first 1000
  overall accuracy: 0.865  (865/1000)
  macro accuracy:   0.867

  Per-class accuracy:
    SUPPORTED           : 0.969 (316/326)
    CONTRADICTED        : 0.913 (305/334)
    NOT MENTIONED       : 0.718 (244/340)

  Confusion matrix (rows=true, cols=pred):
                           SUPPORTED  CONTRADICTED NOT MENTIONED
  SUPPORTED                      316            10             0
  CONTRADICTED                    17           305             0
  NOT MENTIONED                    8             4           244


In [ ]:
print("Baseline Results, first 1000")
baseline_metrics_1000 = compute_metrics(baseline_eval_subset)

Baseline Results, first 1000
  overall accuracy: 0.643  (643/1000)
  macro accuracy:   0.647

  Per-class accuracy:
    SUPPORTED           : 0.819 (267/326)
    CONTRADICTED        : 0.967 (323/334)
    NOT MENTIONED       : 0.156 (53/340)

  Confusion matrix (rows=true, cols=pred):
                           SUPPORTED  CONTRADICTED NOT MENTIONED
  SUPPORTED                      267            37            22
  CONTRADICTED                     4           323             6
  NOT MENTIONED                    0           286            53


## Step 16: Compare Baseline vs DPO

In [ ]:
print("Comparison, first 1000")
print(f"Baseline overall acc: {baseline_metrics_1000['overall_accuracy']:.3f}")
print(f"DPO overall acc:      {dpo_metrics_1000['overall_accuracy']:.3f}")
print(f"Baseline macro acc:   {baseline_metrics_1000['macro_accuracy']:.3f}")
print(f"DPO macro acc:        {dpo_metrics_1000['macro_accuracy']:.3f}")
print(f"Delta overall acc:    {dpo_metrics_1000['overall_accuracy'] - baseline_metrics_1000['overall_accuracy']:+.3f}")
print(f"Delta macro acc:      {dpo_metrics_1000['macro_accuracy'] - baseline_metrics_1000['macro_accuracy']:+.3f}")

Comparison, first 1000
Baseline overall acc: 0.643
DPO overall acc:      0.865
Baseline macro acc:   0.647
DPO macro acc:        0.867
Delta overall acc:    +0.222
Delta macro acc:      +0.219


## Step 17: Qualitative Examples

Pick a few examples where DPO changed the prediction or improved the justification.

In [ ]:
baseline_by_id = {r["id"]: r for r in baseline_eval_subset}
dpo_by_id = {r["id"]: r for r in dpo_eval_subset}

changed = []

for ex in eval_sample[:EVAL_N]:
    ex_id = ex["id"]
    b = baseline_by_id.get(ex_id)
    d = dpo_by_id.get(ex_id)

    if not b or not d:
        continue

    if b["pred_label"] != d["pred_label"]:
        changed.append({
            "id": ex_id,
            "gold": ex["label"],
            "baseline_pred": b["pred_label"],
            "dpo_pred": d["pred_label"],
            "claim": ex["claim"],
            "baseline_raw": b["raw"],
            "dpo_raw": d["raw"],
        })

print("Changed predictions:", len(changed))

for row in changed[:5]:
  if row["gold"] == row["dpo_pred"]:
    print("\n---")
    print("ID:", row["id"])
    print("Gold:", row["gold"])
    print("Baseline:", row["baseline_pred"])
    print("DPO:", row["dpo_pred"])
    print("Claim:", row["claim"])
    print("Baseline raw:", row["baseline_raw"])
    print("DPO raw:", row["dpo_raw"])

Changed predictions: 374

---
ID: 113116
Gold: NOT MENTIONED
Baseline: CONTRADICTED
DPO: NOT MENTIONED
Claim: Renato Balestra was born in a plane.
Baseline raw: CONTRADICTED: The passage provides information about Wenceslaus of Żagań and does not mention Renato Balestra or his birthplace.
DPO raw: NOT MENTIONED: The passage only provides information about Wenceslaus of Żagań and his historical timeline, and does not mention Renato Balestra or any details about his birth or birthplace.

---
ID: 184080
Gold: NOT MENTIONED
Baseline: CONTRADICTED
DPO: NOT MENTIONED
Claim: Kenneth Lonergan is a songwriter.
Baseline raw: CONTRADICTED: The passage mentions that Andreyev's brother V.A. Andreyev was a sculptor, and there is no information about Kenneth Lonergan being a songwriter.
DPO raw: NOT MENTIONED: The passage only states that V.A. Andreyev was a sculptor and does not mention Kenneth Lonergan as a songwriter.


## Additional Step: Hyperparameter Tuning

In [ ]:
# DPO hyperparameter tuning jobs
DPO_CONFIGS = [
    {
        "config_id": "dpo_c2",
        "learning_rate_multiplier": 2.0,
        "n_epochs": 1,
        "batch_size": 1,
        "notes": "Higher learning rate",
    },
    {
        "config_id": "dpo_c3",
        "learning_rate_multiplier": 1.0,
        "n_epochs": 3,
        "batch_size": 1,
        "notes": "Increased training duration",
    },
    {
        "config_id": "dpo_c4",
        "learning_rate_multiplier": 1.0,
        "n_epochs": 1,
        "batch_size": 4,
        "notes": "Increased batch size",
    },
]

dpo_jobs = {}

for cfg in DPO_CONFIGS:
    print("=" * 80)
    print(f"Creating DPO job: {cfg['config_id']}")
    print("Notes:", cfg["notes"])

    job = openai_client.fine_tuning.jobs.create(
        training_file=train_file_id,
        model=BASE_MODEL,
        method={
            "type": "dpo",
            "dpo": {
                "hyperparameters": {
                    "n_epochs": cfg["n_epochs"],
                    "batch_size": cfg["batch_size"],
                    "learning_rate_multiplier": cfg["learning_rate_multiplier"],
                }
            }
        },
        extra_body={"trainingType": "GlobalStandard"},
        suffix=cfg["config_id"],
    )

    dpo_jobs[cfg["config_id"]] = {
        "job_id": job.id,
        "status": job.status,
        "config": cfg,
    }

    print("Job ID:", job.id)
    print("Initial status:", job.status)

Creating DPO job: dpo_c2
Notes: Higher learning rate
Job ID: ftjob-ac11598ad2c14758b395bc5860c18043
Initial status: pending
Creating DPO job: dpo_c3
Notes: Increased training duration
Job ID: ftjob-fd0b7badeffd45c4acdd5641a4c659ed
Initial status: pending
Creating DPO job: dpo_c4
Notes: Increased batch size
Job ID: ftjob-e163528e6cf34e8bb6abe72dc7b1de08
Initial status: pending


In [ ]:
# Save job ids

with open("dpo_tuning_jobs.json", "w") as f:
    json.dump(dpo_jobs, f, indent=2)

print(json.dumps(dpo_jobs, indent=2))

NameError: name 'json' is not defined

In [ ]:
# Monitor Jobs
import time

def retrieve_job(job_id):
    return openai_client.fine_tuning.jobs.retrieve(job_id)

def monitor_dpo_jobs(dpo_jobs, poll_seconds=60):
    unfinished = set(dpo_jobs.keys())
    final_jobs = {}

    while unfinished:
        print("\nChecking jobs...")
        finished_now = []

        for config_id in list(unfinished):
            job_id = dpo_jobs[config_id]["job_id"]
            current = retrieve_job(job_id)

            print(f"{config_id} | {job_id} | {current.status}")

            if current.status in {"succeeded", "failed", "cancelled"}:
                final_jobs[config_id] = current
                finished_now.append(config_id)

        for config_id in finished_now:
            unfinished.remove(config_id)

        if unfinished:
            time.sleep(poll_seconds)

    return final_jobs

final_dpo_jobs = monitor_dpo_jobs(dpo_jobs, poll_seconds=60)


Checking jobs...
dpo_c4 | ftjob-e163528e6cf34e8bb6abe72dc7b1de08 | pending
dpo_c3 | ftjob-fd0b7badeffd45c4acdd5641a4c659ed | pending
dpo_c2 | ftjob-ac11598ad2c14758b395bc5860c18043 | pending

Checking jobs...
dpo_c4 | ftjob-e163528e6cf34e8bb6abe72dc7b1de08 | pending
dpo_c3 | ftjob-fd0b7badeffd45c4acdd5641a4c659ed | pending
dpo_c2 | ftjob-ac11598ad2c14758b395bc5860c18043 | pending

Checking jobs...
dpo_c4 | ftjob-e163528e6cf34e8bb6abe72dc7b1de08 | pending
dpo_c3 | ftjob-fd0b7badeffd45c4acdd5641a4c659ed | pending
dpo_c2 | ftjob-ac11598ad2c14758b395bc5860c18043 | pending

Checking jobs...
dpo_c4 | ftjob-e163528e6cf34e8bb6abe72dc7b1de08 | pending
dpo_c3 | ftjob-fd0b7badeffd45c4acdd5641a4c659ed | pending
dpo_c2 | ftjob-ac11598ad2c14758b395bc5860c18043 | pending

Checking jobs...
dpo_c4 | ftjob-e163528e6cf34e8bb6abe72dc7b1de08 | pending
dpo_c3 | ftjob-fd0b7badeffd45c4acdd5641a4c659ed | pending
dpo_c2 | ftjob-ac11598ad2c14758b395bc5860c18043 | pending

Checking jobs...
dpo_c4 | ftjob-e163528

In [ ]:
# Print final model names:
for config_id, job in final_dpo_jobs.items():
    print("=" * 80)
    print("Config:", config_id)
    print("Status:", job.status)
    print("Fine-tuned model:", getattr(job, "fine_tuned_model", None))
    print("Error:", getattr(job, "error", None))

NameError: name 'final_dpo_jobs' is not defined

In [ ]:
# Need to run on test set

## Step 18: Delete Deployment

Avoid ongoing hosting cost.

# Hyperparameter Checking

In [54]:
DPO_DEPLOYMENTS = {
    "dpo_c1": "1-nano-2025-04-14-dpo_c1",
    "dpo_c4": "1-nano-2025-04-14-dpo_c4",
}

SYSTEM_PROMPT = """You are a fact-checking assistant. Given a passage and a claim, respond with the verdict followed by a one-sentence justification quoting or closely paraphrasing the passage.
Format: LABEL: justification sentence
Label must be one of: SUPPORTED, CONTRADICTED, NOT MENTIONED"""

VALID_LABELS = {"SUPPORTED", "CONTRADICTED", "NOT MENTIONED"}

In [66]:
# Build stratified dev sample
dev_data = load_jsonl(dev_path)

buckets = defaultdict(list)
for ex in dev_data:
    buckets[ex["label"]].append(ex)

eval_sample = []
for lbl, items in buckets.items():
    random.shuffle(items)
    eval_sample.extend(items[:334])

random.shuffle(eval_sample)

print(f"Dev sample: {len(eval_sample)} examples")
print(Counter(ex["label"] for ex in eval_sample))

Dev sample: 1002 examples
Counter({'NOT MENTIONED': 334, 'SUPPORTED': 334, 'CONTRADICTED': 334})


In [67]:
VALID_LABELS = {"SUPPORTED", "CONTRADICTED", "NOT MENTIONED"}

def extract_label(text):
    for lbl in VALID_LABELS:
        if text.upper().startswith(lbl + ":"):
            return lbl
    if text.strip() in VALID_LABELS:
        return text.strip()
    for lbl in sorted(VALID_LABELS, key=len, reverse=True):
        if re.search(rf'\b{lbl}\b', text.upper()):
            return lbl
    return None

def predict_dpo(passage, claim, deployment, temperature=0.0):
    user_msg = f"Passage: {passage}\n\nClaim: {claim}"
    for attempt in range(3):
        try:
            resp = openai_client.chat.completions.create(
                model=deployment,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user",   "content": user_msg},
                ],
                temperature=temperature,
                max_tokens=150,
            )
            text = resp.choices[0].message.content.strip()
            label = extract_label(text)
            if label:
                return label, text
        except Exception as e:
            if attempt < 2:
                time.sleep(2 ** attempt)
            else:
                print(f"  failed after 3 attempts: {e}")
    return None, None

def run_eval(cfg_name, deployment, examples, out_path):
    done_ids = set()
    results = []
    if os.path.exists(out_path):
        with open(out_path) as f:
            for line in f:
                r = json.loads(line)
                done_ids.add(r["id"])
                results.append(r)
        print(f"resuming, {len(done_ids)} already done")

    skipped = 0
    with open(out_path, "a") as out_f:
        for i, ex in enumerate(examples):
            if ex["id"] in done_ids:
                continue
            pred_label, raw = predict_dpo(ex["passage"], ex["claim"], deployment)
            if pred_label is None:
                skipped += 1
                continue
            record = {"id": ex["id"], "label": ex["label"], "pred_label": pred_label, "raw": raw}
            out_f.write(json.dumps(record) + "\n")
            out_f.flush()
            results.append(record)
            if (i + 1) % 100 == 0:
                print(f"  [{i+1}/{len(examples)}] skipped={skipped}")

    print(f"done. total={len(results)} skipped={skipped}")
    return results

In [58]:
test_ex = eval_sample[0]

pred_label, raw = predict_dpo(
    test_ex["passage"],
    test_ex["claim"],
    "1-nano-2025-04-14-dpo_c4"
)

print("pred_label:", pred_label)
print("raw:", raw)

pred_label: SUPPORTED
raw: SUPPORTED: Passage states that "It ranks as the highest-grossing animated film of all time," directly supporting the claim.


In [65]:
# Run inference only on c4

cfg_name = "dpo_c4"
deployment = DPO_DEPLOYMENTS[cfg_name]
out_path = f"{data_dir}/{cfg_name}_dev.jsonl"

print(f"Config: {cfg_name}")
print(f"Deployment: {deployment}")
print(f"Output path: {out_path}")

dpo_c4_results = run_eval(
    cfg_name=cfg_name,
    deployment=deployment,
    examples=eval_sample,
    out_path=out_path,
)

Config: dpo_c4
Deployment: 1-nano-2025-04-14-dpo_c4
Output path: /content/drive/MyDrive/dpo_c4_dev.jsonl
  failed after 3 attempts: Error code: 400 - {'error': {'message': "The response was filtered due to the prompt triggering Azure OpenAI's content management policy. Please modify your prompt and retry. To learn more about our content filtering policies please read our documentation: https://go.microsoft.com/fwlink/?linkid=2198766", 'type': None, 'param': 'prompt', 'code': 'content_filter', 'status': 400, 'innererror': {'code': 'ResponsibleAIPolicyViolation', 'content_filter_result': {'hate': {'filtered': True, 'severity': 'medium'}, 'jailbreak': {'detected': False, 'filtered': False}, 'self_harm': {'filtered': False, 'severity': 'safe'}, 'sexual': {'filtered': False, 'severity': 'safe'}, 'violence': {'filtered': False, 'severity': 'low'}}}}}
  [300/1002] skipped=63
  [500/1002] skipped=93
  [600/1002] skipped=114
  [700/1002] skipped=134
  [800/1002] skipped=148
  failed after 3 att

In [71]:
def compute_metrics(results):
    labels = ["SUPPORTED", "CONTRADICTED", "NOT MENTIONED"]
    per_class = {lbl: {"correct": 0, "total": 0} for lbl in labels}
    confusion = {true: {pred: 0 for pred in labels} for true in labels}
    total_correct = 0

    for r in results:
        gt, pred = r["label"], r["pred_label"]
        per_class[gt]["total"] += 1
        if pred in labels:
            confusion[gt][pred] += 1
        if pred == gt:
            per_class[gt]["correct"] += 1
            total_correct += 1

    macro_acc = sum(
        per_class[lbl]["correct"] / per_class[lbl]["total"]
        for lbl in labels if per_class[lbl]["total"] > 0
    ) / len(labels)
    overall_acc = total_correct / len(results)

    print(f"  overall accuracy: {overall_acc:.3f}  ({total_correct}/{len(results)})")
    print(f"  macro accuracy:   {macro_acc:.3f}")
    print()
    print("  Per-class accuracy:")
    for lbl in labels:
        c = per_class[lbl]
        acc = c["correct"] / c["total"] if c["total"] else 0
        print(f"    {lbl:20s}: {acc:.3f}  ({c['correct']}/{c['total']})")
    print()
    print("  Confusion matrix (rows=true, cols=pred):")
    col_w = 14
    print(" " * 22 + "".join(f"{lbl[:col_w]:>{col_w}}" for lbl in labels))
    for true_lbl in labels:
        row = f"  {true_lbl:20s}" + "".join(
            f"{confusion[true_lbl][pred_lbl]:>{col_w}}" for pred_lbl in labels
        )
        print(row)

    return macro_acc

# reload from disk so cell is self-contained
all_results = {}
for cfg_name in DPO_DEPLOYMENTS:
    path = f"{data_dir}/{cfg_name}_dev.jsonl"
    if os.path.exists(path):
        records = []
        with open(path) as f:
            for line in f:
                if not line.strip(): continue
                try:
                    records.append(json.loads(line))
                except json.JSONDecodeError:
                    pass
        all_results[cfg_name] = records
        print(f"loaded {cfg_name}: {len(records)} records")
    else:
        print(f"missing: {path}")

print("\nDPO Dev Results\n")
best_cfg, best_acc = None, 0
for cfg_name, results in all_results.items():
    print(f"Config: {cfg_name}")
    acc = compute_metrics(results)
    if acc > best_acc:
        best_acc, best_cfg = acc, cfg_name
    print()

print(f"Best config: {best_cfg}  (macro accuracy={best_acc:.3f})")

loaded dpo_c1: 1029 records
loaded dpo_c4: 815 records

DPO Dev Results

Config: dpo_c1
  overall accuracy: 0.942  (969/1029)
  macro accuracy:   0.933

  Per-class accuracy:
    SUPPORTED           : 0.947  (355/375)
    CONTRADICTED        : 0.856  (232/271)
    NOT MENTIONED       : 0.997  (382/383)

  Confusion matrix (rows=true, cols=pred):
                           SUPPORTED  CONTRADICTED NOT MENTIONED
  SUPPORTED                      355            13             7
  CONTRADICTED                    36           232             3
  NOT MENTIONED                    1             0           382

Config: dpo_c4
  overall accuracy: 0.929  (757/815)
  macro accuracy:   0.904

  Per-class accuracy:
    SUPPORTED           : 0.979  (326/333)
    CONTRADICTED        : 0.940  (312/332)
    NOT MENTIONED       : 0.793  (119/150)

  Confusion matrix (rows=true, cols=pred):
                           SUPPORTED  CONTRADICTED NOT MENTIONED
  SUPPORTED                      326             7  

In [70]:
# Run inference only on c4

cfg_name = "dpo_c1"
deployment = DPO_DEPLOYMENTS[cfg_name]
out_path = f"{data_dir}/{cfg_name}_dev.jsonl"

print(f"Config: {cfg_name}")
print(f"Deployment: {deployment}")
print(f"Output path: {out_path}")

dpo_c1_results = run_eval(
    cfg_name=cfg_name,
    deployment=deployment,
    examples=eval_sample,
    out_path=out_path,
)

Config: dpo_c1
Deployment: 1-nano-2025-04-14-dpo_c1
Output path: /content/drive/MyDrive/dpo_c1_dev.jsonl
resuming, 242 already done
  [200/1002] skipped=15
  [300/1002] skipped=26
  failed after 3 attempts: Error code: 429 - {'error': {'message': 'Too Many Requests', 'type': 'too_many_requests', 'param': None, 'code': 'too_many_requests'}}
  failed after 3 attempts: Error code: 429 - {'error': {'message': 'Too Many Requests', 'type': 'too_many_requests', 'param': None, 'code': 'too_many_requests'}}
  [400/1002] skipped=38
  failed after 3 attempts: Error code: 400 - {'error': {'message': "The response was filtered due to the prompt triggering Azure OpenAI's content management policy. Please modify your prompt and retry. To learn more about our content filtering policies please read our documentation: https://go.microsoft.com/fwlink/?linkid=2198766", 'type': None, 'param': 'prompt', 'code': 'content_filter', 'status': 400, 'innererror': {'code': 'ResponsibleAIPolicyViolation', 'content_